# Experiment 03: Negative Controls

Specificity checks for the alignment metrics. Includes artificial off-target subspaces and placeholders for time-shuffled or random-target training controls.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


Repo root: C:\Users\WWindows10\Documents\github_project\python-neural-network-research


In [2]:
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []

from src.metrics import (
    build_sampled_basis_matrix_from_indices,
    calculate_rank_metrics,
    calculate_subspace_alignment_from_matrices,
)


In [3]:
NUM_FREQS = 4
SEQ_LEN = 2048
TARGET_INDICES = np.arange(SEQ_LEN, dtype=np.int64)
TRUE_THETAS = tuple(np.linspace(0.10 * np.pi, 0.70 * np.pi, NUM_FREQS))
WRONG_THETAS = tuple(np.linspace(0.15 * np.pi, 0.75 * np.pi, NUM_FREQS))
F_true = build_sampled_basis_matrix_from_indices(time_mode="discrete", target_indices=TARGET_INDICES, thetas=TRUE_THETAS)
F_wrong = build_sampled_basis_matrix_from_indices(time_mode="discrete", target_indices=TARGET_INDICES, thetas=WRONG_THETAS)
Qf_true, _ = np.linalg.qr(F_true)
P_perp = np.eye(Qf_true.shape[0]) - Qf_true @ Qf_true.T
rng = np.random.default_rng(123)


In [4]:
control_cases = {
    "same_subspace": F_true @ rng.standard_normal((F_true.shape[1], F_true.shape[1])),
    "wrong_basis": F_wrong @ rng.standard_normal((F_wrong.shape[1], F_wrong.shape[1])),
    "random": rng.standard_normal((F_true.shape[0], F_true.shape[1])),
    "random_perp": P_perp @ rng.standard_normal((F_true.shape[0], F_true.shape[1])),
}

records = []
for case_name, H in tqdm(list(control_cases.items()), desc="experiment 3 controls"):
    metrics = calculate_subspace_alignment_from_matrices(H=H, F=F_true, theory_dim=2 * NUM_FREQS)
    singular_values = np.linalg.svd(H, full_matrices=False, compute_uv=False)
    rank_metrics = calculate_rank_metrics(singular_values, threshold=0.05)
    records.append({
        "case": case_name,
        "rank_threshold": rank_metrics["rank_threshold"],
        "rank_entropy": rank_metrics["rank_entropy"],
        "h_numerical_dim": metrics["h_numerical_dim"],
        "test_align_coverage_full": metrics["align_coverage_full"],
        "test_recon_r2_qf_from_h": metrics["recon_r2_qf_from_h"],
        "test_align_mean_angle_deg_full": metrics["align_mean_angle_deg_full"],
    })

pd.DataFrame(records)


,case,test_align_coverage_full,test_recon_r2_qf_from_h,test_align_mean_angle_deg_full
0,same_subspace,1.000000e+00,1.000000,4.672065e-07
1,wrong_basis,1.756835e-05,0.000018,8.977912e+01
2,random,3.948633e-03,0.003949,8.682473e+01
3,random_perp,1.098862e-32,0.000000,9.000000e+01


## Optional training-based controls

Add manual cells below for:
- time-shuffled targets
- random signals
- mismatched basis evaluation after training

The core metric-specificity sanity checks above can be executed first.

In [ ]:
detailed_records = []
for case_name, H in tqdm(list(control_cases.items()), desc="experiment 3 detailed metrics"):
    metrics = calculate_subspace_alignment_from_matrices(H=H, F=F_true, theory_dim=2 * NUM_FREQS)
    singular_values = np.linalg.svd(H, full_matrices=False, compute_uv=False)
    rank_metrics = calculate_rank_metrics(singular_values, threshold=0.05)
    row = {
        "case": case_name,
        "rank_threshold": rank_metrics["rank_threshold"],
        "rank_entropy": rank_metrics["rank_entropy"],
    }
    for key, value in metrics.items():
        if np.isscalar(value):
            row[key] = value
    detailed_records.append(row)

control_metrics_df = pd.DataFrame(detailed_records)
control_metrics_df
